# Final Evaluation — Test Set Unlock

**This notebook runs exactly once.**

All model selection, hyperparameter tuning, and architecture decisions were made using 5-fold CV on the 148 training samples. The test set (38 samples) was never touched. This notebook is the single point where we evaluate all models on held-out data and report final numbers.

**What we report:**
- AUC-ROC — primary metric, threshold-free
- Sensitivity (recall for cancer) and Specificity at threshold = 0.5
- ROC curves for all four models on one plot

**Why threshold = 0.5 for sensitivity/specificity:**  
We did not tune the threshold on training data, so 0.5 is the honest default. In a real clinical deployment you would tune the threshold on a separate calibration set — that is out of scope here.

**Models being evaluated:**
| Model | File | Features |
|-------|------|----------|
| Random Forest | `models/rf_final.pkl` | 18 tabular |
| XGBoost | `models/xgb_final.pkl` | 18 tabular |
| MLP | `models/mlp_final.pkl` | 18 tabular |
| 1D CNN | `models/cnn_final.pt` | 311 histogram |

## Step 1 — Load test data + all models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, pathlib, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

ROOT      = pathlib.Path().resolve().parent
SPLIT_DIR = ROOT / 'data' / 'processed' / 'split'

# ── Tabular test set (18 features) ──────────────────────────────────────────
X_test   = np.load(SPLIT_DIR / 'X_test.npy')
y_test   = np.load(SPLIT_DIR / 'y_test.npy')

# ── Histogram test set (311 features) for CNN ───────────────────────────────
df        = pd.read_csv(ROOT / 'data' / 'processed' / 'features_model.csv')
hist_cols = sorted([c for c in df.columns if c.startswith('hist_')])
X_hist    = df[hist_cols].astype(float).values
y_all     = (df['label'] == 'cancer').astype(int).values

_, X_test_h, _, y_test_h = train_test_split(
    X_hist, y_all, test_size=0.20, stratify=y_all, random_state=42
)
with open(ROOT / 'models' / 'scaler_hist.pkl', 'rb') as f:
    scaler_hist = pickle.load(f)
X_test_h = scaler_hist.transform(X_test_h)

# ── Load tabular models ──────────────────────────────────────────────────────
with open(ROOT / 'models' / 'rf_final.pkl',  'rb') as f: rf_model  = pickle.load(f)
with open(ROOT / 'models' / 'xgb_final.pkl', 'rb') as f: xgb_model = pickle.load(f)
with open(ROOT / 'models' / 'mlp_final.pkl', 'rb') as f: mlp_model = pickle.load(f)

# ── Load CNN ─────────────────────────────────────────────────────────────────
class CfDNA_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 8,  kernel_size=15, padding=7), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(8, 16, kernel_size=7,  padding=3), nn.ReLU(), nn.MaxPool1d(4),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(304, 32), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(32, 1),
        )
    def forward(self, x): return self.fc(self.conv(x)).squeeze(1)

cnn_model = CfDNA_CNN()
cnn_model.load_state_dict(torch.load(ROOT / 'models' / 'cnn_final.pt', weights_only=True))
cnn_model.eval()

print(f'Test set : {X_test.shape[0]} samples  '
      f'(healthy={(y_test==0).sum()}  cancer={(y_test==1).sum()})')
print('All models loaded.')

## Step 2 — Predictions + metrics

In [ ]:
def sens_spec(y_true, proba, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return sensitivity, specificity

# ── Probabilities ────────────────────────────────────────────────────────────
prob_rf  = rf_model.predict_proba(X_test)[:, 1]
prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
prob_mlp = mlp_model.predict_proba(X_test)[:, 1]

with torch.no_grad():
    logits_cnn = cnn_model(torch.tensor(X_test_h[:, None, :], dtype=torch.float32))
prob_cnn = torch.sigmoid(logits_cnn).numpy()

# ── Metrics ──────────────────────────────────────────────────────────────────
models = {
    'Random Forest' : (prob_rf,  y_test),
    'XGBoost'       : (prob_xgb, y_test),
    'MLP'           : (prob_mlp, y_test),
    '1D CNN'        : (prob_cnn, y_test_h),
}

rows = []
for name, (prob, y) in models.items():
    auc          = roc_auc_score(y, prob)
    sens, spec   = sens_spec(y, prob, threshold=0.5)
    rows.append({'Model': name, 'Test AUC': round(auc, 4),
                 'Sensitivity': round(sens, 4), 'Specificity': round(spec, 4)})

results_df = pd.DataFrame(rows).sort_values('Test AUC', ascending=False).reset_index(drop=True)

print('=' * 55)
print('FINAL TEST SET RESULTS')
print('=' * 55)
print(results_df.to_string(index=False))
print('=' * 55)
print()
print('Sensitivity = cancer recall  (TP / (TP + FN))')
print('Specificity = healthy recall (TN / (TN + FP))')
print('Threshold = 0.5 for sensitivity/specificity')

## Step 3 — ROC curves